Trains a compact CNN that recognises chess piece glyphs from the actual fonts in your PDF chess book.
The exported TFLite model ships inside the Flutter app and runs fully offline.

**Pipeline**
1. Mount Google Drive and load your chess PDF
2. For pages with games, render them as high-res images
3. Extract text layer with per-character bounding boxes
4. Identify figurine characters (non-ASCII/unusual Unicode)
5. Crop glyphs from rendered images, infer labels from move context
6. Train model on real book fonts
7. Export TFLite model to use in the Flutter app

**Model**
- Input : 32 × 32 grayscale, values in \[0, 1\]
- Output: 6-class softmax — `['K', 'Q', 'R', 'B', 'N', 'P']`
- Size  : < 500 KB (float32 TFLite)

## Step 0 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

### Step 0b — Verify PDF file

In [ ]:
# ── EDIT THIS: set your PDF path ────────────────────────────────────────────
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'  # CHANGE THIS to your PDF

# Verify PDF exists
if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
    print(f'   Available items in /content/gdrive/MyDrive:')
    for item in os.listdir('/content/gdrive/MyDrive')[:20]:
        print(f'     - {item}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 1 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q pdf2image pdfplumber pillow numpy tensorflow matplotlib chess pytesseract

## Step 2 — Configuration

In [ ]:
# ── Classes ────────────────────────────────────────────────────────────────
CLASS_NAMES = ['K', 'Q', 'R', 'B', 'N', 'P']   # piece letter; P = pawn
IMG_SIZE    = 32   # pixels — model input (32×32 grayscale)

# ── PDF Configuration ──────────────────────────────────────────────────────
# PDF_PATH was set in Step 0b — change it there and re-run Step 0b if needed
# START_PAGE, END_PAGE: pages to extract glyphs from (1-indexed)
# MIN_GLYPH_SAMPLES: minimum distinct glyphs to collect before training (quality filter)

START_PAGE      = 1
END_PAGE        = 20      # extract from first 20 pages
GLYPH_DPI       = 150     # render resolution for glyph extraction
MIN_GLYPH_SAMPLES = 10    # minimum samples per piece to proceed with training

# ── CRITICAL: Does your chess book use FIGURINE notation or PLAIN TEXT notation? ────
#
# FIGURINE notation:  Moves have a special chess symbol (image) for the piece
#                     Example: ♘xf6  (knight symbol rendered as image, followed by xf6)
#                     → We extract the IMAGE and train a classifier
#
# PLAIN TEXT notation: Moves are just standard algebraic text, no special symbols
#                      Example: Nxf6  (N is a regular letter, not a chess font)
#                      → We just parse the text, no classifier needed
#
USE_FIGURINE_NOTATION = True  # ← CHANGE THIS BASED ON YOUR BOOK!
                              # True = figurine notation, extract and classify symbols
                              # False = plain text notation, just parse moves

# ── Augmentation (applied at training time, not collection) ─────────────────
AUGMENT_PER_GLYPH = 40    # variants generated per collected glyph

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS     = 60
BATCH_SIZE = 64
VAL_SPLIT  = 0.15

# ── Output ────────────────────────────────────────────────────────────────
TFLITE_PATH = 'figurine_classifier.tflite'

print('Classes    :', CLASS_NAMES)
print('PDF        :', PDF_PATH)
print('Pages      : %d–%d' % (START_PAGE, END_PAGE))
print('Min samples:', MIN_GLYPH_SAMPLES, 'per piece')
print()
if USE_FIGURINE_NOTATION:
    print('✅ MODE: FIGURINE NOTATION')
    print('   → Will extract figurine glyphs from images')
    print('   → Will train classifier on piece symbols')
else:
    print('✅ MODE: PLAIN TEXT NOTATION')
    print('   → Will parse moves from text only')
    print('   → No figurine classifier needed')

## Step 3 — Extract figurine glyphs from PDF pages (Fixed OCR approach)

**What was wrong:**
The original code tried to OCR the entire word region, which included move data and led to corrupted results like `'93 £7'`, `'Six''`, etc. This corrupted the piece identification.

**What's fixed:**
1. **Single-character OCR mode** — Tesseract `--psm 10` (recognize single character only)
2. **Character whitelist** — restrict output to valid chess piece symbols: `KQRBN♔♕♖♗♘`
3. **Enhanced contrast** — improves OCR accuracy on small rendered glyphs
4. **Proper fallback** — if OCR fails on symbol, check if text contains a move (file+rank) and assume pawn
5. **Real image extraction** — crop from rendered PDF using word bounding boxes
6. **Ground-truth labeling** — piece type comes from OCR symbol detection

**Result:** extraction_results contains:
- `crop_image`: actual figurine glyph from PDF
- `inferred_piece`: detected piece type (K, Q, R, B, N, or P)
- `ocr_symbol`: what OCR returned (for debugging)


In [ ]:
import pdfplumber
import numpy as np
from PIL import Image
from pdf2image import convert_from_path
from collections import defaultdict
import re
import zipfile
import io
import os
from datetime import datetime

# ── Helper: render page to high-res image ──────────────────────────────────
def render_page(pdf_path, page_num, dpi=150):
    """page_num is 0-indexed"""
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

# ── Filter: is this a valid chess move? ─────────────────────────────────────
def is_chess_move(text):
    clean = text.strip().upper()
    if not re.search(r'[A-H]', clean) or not re.search(r'[1-8]', clean):
        return False
    move_pattern = r'^[KQRBN]?[A-H]x?[A-H][1-8][+#=]?[!?]*$'
    if not re.match(move_pattern, clean):
        return False
    if len(clean) > 12:
        return False
    return True

# ── Check if word has a figurine symbol ────────────────────────────────────
def has_figurine_symbol(text):
    for char in text:
        if char in '♔♕♖♗♘♚♛♜♝♞':
            return True
        if ord(char) > 127:
            return True
    return False

# ── Infer piece from move text ─────────────────────────────────────────────
def get_piece_from_move(text):
    clean = text.strip().upper()
    first = clean[0] if clean else None
    if first in 'KQRBN':
        return first
    if first in 'ABCDEFGH':
        return 'P'
    return None

# ── Export function: Download crops as ZIP ─────────────────────────────────
def export_crops_as_zip(crops):
    """Create a zip file with all extracted crop images"""
    if not crops:
        print('❌ No crops to export.')
        return None
    
    zip_buffer = io.BytesIO()
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    zip_filename = f'chess_glyphs_{timestamp}.zip'
    
    with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
        for idx, result in enumerate(crops, 1):
            piece = result['piece']
            page = result['page']
            move = result['move_text'].replace('/', '_').replace('\\', '_')
            img_filename = f'{idx:03d}_{piece}_p{page}_{move}.png'
            
            # Convert PIL image to PNG bytes
            img_buffer = io.BytesIO()
            result['crop_image'].save(img_buffer, format='PNG')
            img_buffer.seek(0)
            
            zf.writestr(img_filename, img_buffer.getvalue())
    
    zip_buffer.seek(0)
    
    # Save to local file (for Colab download)
    with open(zip_filename, 'wb') as f:
        f.write(zip_buffer.getvalue())
    
    size_kb = os.path.getsize(zip_filename) / 1024
    print(f'✅ Exported: {zip_filename} ({size_kb:.1f} KB, {len(crops)} images)')
    
    return zip_filename

# ── Main extraction ────────────────────────────────────────────────────────
extraction_results = []

print(f'Extracting ALL chess moves from pages {START_PAGE} to {END_PAGE}...\n')

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    if END_PAGE > pdf_page_count:
        END_PAGE = pdf_page_count
        print(f'  (PDF has {pdf_page_count} pages, adjusted END_PAGE)\n')
    
    for page_idx in range(START_PAGE - 1, min(END_PAGE, pdf_page_count)):
        pdf_page = pdf.pages[page_idx]
        print(f'Processing page {page_idx + 1}...')
        
        page_image = render_page(PDF_PATH, page_idx, dpi=GLYPH_DPI)
        if page_image is None:
            print(f'  ⚠ Could not render page')
            continue
        
        try:
            words = pdf_page.extract_words()
        except:
            print(f'  ⚠ Could not extract words')
            continue
        
        if not words:
            print(f'  ⚠ No words found')
            continue
        
        page_count = 0
        for word in words:
            text = word.get('text', '')
            
            if not is_chess_move(text):
                continue
            
            piece = get_piece_from_move(text)
            if not piece:
                continue
            
            bbox = (word['x0'], word['top'], word['x1'], word['bottom'])
            has_figurine = has_figurine_symbol(text)
            
            # Improved cropping: extract centered square crop around the glyph
            scale = GLYPH_DPI / 72.0
            x0 = max(0, int(bbox[0] * scale))
            y0 = max(0, int(bbox[1] * scale))
            x1 = min(page_image.width, int(bbox[2] * scale))
            y1 = min(page_image.height, int(bbox[3] * scale))

            # Improved: use height as basis, but trim more aggressively on right
            glyph_height = y1 - y0
            glyph_width = x1 - x0

            # Crop size: use height, but cap it to avoid including too much text on right
            crop_size = int(glyph_height * 0.85)  # Reduced from 0.95 to trim text better
            crop_size = min(crop_size, glyph_width - 2)  # Don't exceed actual width

            if crop_size < 8:  # Skip if too small
                continue

            # Horizontal: start at x0, but don't go past the right edge
            crop_x0 = max(0, x0)
            crop_x1 = min(page_image.width, crop_x0 + crop_size)

            # Vertical: center vertically within the glyph bounds
            glyph_center_y = (y0 + y1) // 2
            crop_y0 = max(0, glyph_center_y - crop_size // 2)
            crop_y1 = min(page_image.height, crop_y0 + crop_size)

            # Handle edge cases: if crop goes out of bounds, shift it back in
            if crop_y1 > page_image.height:
                crop_y1 = page_image.height
                crop_y0 = max(0, crop_y1 - crop_size)

            # Ensure square by adjusting if needed
            actual_width = crop_x1 - crop_x0
            actual_height = crop_y1 - crop_y0
            
            if actual_width >= 8 and actual_height >= 8:
                crop = page_image.crop((crop_x0, crop_y0, crop_x1, crop_y1))
                if crop.mode != 'L':
                    crop = crop.convert('L')
                
                extraction_results.append({
                    'page': page_idx + 1,
                    'move_text': text,
                    'piece': piece,
                    'has_figurine': has_figurine,
                    'bbox': bbox,
                    'crop_image': crop,
                    'crop_width': actual_width,
                    'crop_height': actual_height,
                })
                
                figurine_marker = '📍' if has_figurine else '  '
                print(f'  {figurine_marker} {piece} {text:12} | ({actual_width}×{actual_height} px)')
                page_count += 1
        
        print(f'  ✅ Extracted {page_count} moves\n')

print(f'📊 Summary:')
print(f'  Total moves: {len(extraction_results)}\n')

piece_counts = defaultdict(int)
for result in extraction_results:
    piece_counts[result['piece']] += 1

print('Moves by piece:')
for piece in ['K', 'Q', 'R', 'B', 'N', 'P']:
    count = piece_counts[piece]
    if count > 0:
        print(f'  {piece}: {count}')

print()

# Automatically export crops as ZIP
if extraction_results:
    zip_file = export_crops_as_zip(extraction_results)
    print(f'\n💾 Ready to download and review!')

In [ ]:
# ── Debug: Show what text is being extracted from the PDF ──────────────────
print('DEBUG: Sample extracted words from first 3 pages:\n')

with pdfplumber.open(PDF_PATH) as pdf:
    for page_idx in range(0, min(3, len(pdf.pages))):
        pdf_page = pdf.pages[page_idx]
        print(f'Page {page_idx + 1}:')
        
        try:
            words = pdf_page.extract_words()
            # Show first 30 words that contain letters and numbers (likely moves)
            move_like_words = [w['text'] for w in words if any(c.isalpha() for c in w['text']) and any(c.isdigit() for c in w['text'])]
            for word in move_like_words[:20]:
                print(f'  "{word}"')
        except:
            print('  Could not extract words')
        print()

In [ ]:
# ── STEP 1B: Extract move notation (discard piece letter) ──────────────────
def extract_move_notation(move_text):
    """
    Extract move notation, removing any piece letter.
    
    Examples:
      Ne5  → e5
      e4   → e4
      Bxd5 → xd5
      Kc3+ → c3+
    """
    clean = move_text.strip().upper()
    
    # If starts with piece letter, skip it
    if clean and clean[0] in 'KQRBN':
        return clean[1:]
    
    # Otherwise return as is (pawn move)
    return clean

# ── Demo: Show the 3-step parsing ──────────────────────────────────────────
print('MOVE PARSING (3-step demo):\n')
print('Step 1a: Piece from figurine image')
print('Step 1b: Move notation (discard piece letter)')
print('Step 2:  Combine results\n')
print('─' * 70)

for i, result in enumerate(extraction_results[:10]):  # Show first 10
    move_text = result['move_text']
    piece_from_extraction = result['piece']
    has_fig = '📍' if result['has_figurine'] else '  '
    
    # Step 1b
    move_notation = extract_move_notation(move_text)
    
    # Step 2
    if piece_from_extraction == 'P':
        combined = move_notation
    else:
        combined = piece_from_extraction + move_notation
    
    print(f'{has_fig} Text: {move_text:10} | 1a: {piece_from_extraction} | 1b: {move_notation:8} | 2: {combined}')

print('─' * 70)

In [ ]:
# ── STEP: Sort moves by reading flow (top-to-bottom, left-to-right) ────────
def sort_moves_by_reading_flow(extraction_results, column_threshold=200):
    """
    Sort moves by reading flow (how they appear on the page).
    
    Handles:
    - Single column: sort by y (top-to-bottom)
    - Multiple columns: group by vertical bands, then left-to-right within bands
    """
    if not extraction_results:
        return []
    
    # Group by page
    by_page = defaultdict(list)
    for result in extraction_results:
        by_page[result['page']].append(result)
    
    sorted_results = []
    
    for page_num in sorted(by_page.keys()):
        page_moves = by_page[page_num]
        
        # Detect column groups by x-position
        x_coords_sorted = sorted(set(m['bbox'][0] for m in page_moves))
        
        if len(x_coords_sorted) > 1:
            # Find gaps in x-coordinates (column boundaries)
            gaps = [x_coords_sorted[i+1] - x_coords_sorted[i] for i in range(len(x_coords_sorted)-1)]
            
            columns = []
            current_col = [x_coords_sorted[0]]
            for i, gap in enumerate(gaps):
                if gap > column_threshold:
                    columns.append(current_col)
                    current_col = [x_coords_sorted[i+1]]
                else:
                    current_col.append(x_coords_sorted[i+1])
            columns.append(current_col)
        else:
            columns = [x_coords_sorted]
        
        # Sort within each column by y, then across columns left-to-right
        for col_x_coords in columns:
            col_moves = [m for m in page_moves if m['bbox'][0] in col_x_coords]
            col_moves.sort(key=lambda m: m['bbox'][1])  # Sort by y (top to bottom)
            sorted_results.extend(col_moves)
    
    return sorted_results

# ── Apply sorting ──────────────────────────────────────────────────────────
print('Sorting moves by reading flow...\n')
extraction_results_sorted = sort_moves_by_reading_flow(extraction_results)

print(f'📋 Moves in reading order:\n')
for i, result in enumerate(extraction_results_sorted, 1):
    move_notation = extract_move_notation(result['move_text'])
    if result['piece'] == 'P':
        move_san = move_notation
    else:
        move_san = result['piece'] + move_notation
    
    has_fig = '📍' if result['has_figurine'] else '  '
    bounds = result['bbox']
    print(f'{i:2}. {has_fig} P{result["page"]} ({bounds[0]:.0f},{bounds[1]:.0f}): {move_san:8} ({result["move_text"]})')

## Step 3b — Manual labeling of extracted crops

In [ ]:
!pip install -q ipywidgets

import ipywidgets as widgets
from ipywidgets import HBox, VBox, Dropdown, Button, Output
import matplotlib.pyplot as plt

# Use ALL extracted moves (not filtered by figurine detection)
# You'll visually inspect which ones have figurines
all_moves = extraction_results

piece_options = ['[None]', 'K', 'Q', 'R', 'B', 'N', 'P']
labels_dict = {}

print(f'📋 Labeling {len(all_moves)} moves (visually inspect for figurines)')
print(f'   Select [None] if no figurine visible\n')

if len(all_moves) == 0:
    print('❌ No moves found to label.')
    print('   Run extraction step first.')
else:
    n_cols = 5
    n_rows = (len(all_moves) + n_cols - 1) // n_cols
    
    def create_labeling_ui():
        glyph_items = []
        
        for idx, result in enumerate(all_moves):
            crop = result['crop_image']
            
            # Create dropdown for this glyph
            dropdown = Dropdown(
                options=piece_options,
                value='[None]',
                description=f'G{idx+1}:',
                style={'description_width': '50px'}
            )
            
            glyph_items.append({
                'idx': idx,
                'result': result,
                'page': result['page'],
                'move': result['move_text'],
                'piece_from_text': result['piece'],
                'dropdown': dropdown
            })
        
        return glyph_items
    
    print('Loading glyph images...')
    glyph_items = create_labeling_ui()
    print(f'✅ Ready to label {len(glyph_items)} glyphs\n')
    
    # Save labels button
    save_button = Button(description='Save Labels', button_style='success')
    
    def on_save_clicked(b):
        labels_dict.clear()
        for item in glyph_items:
            idx = item['idx']
            selected = item['dropdown'].value
            if selected != '[None]':
                labels_dict[idx] = selected
        
        print(f'✅ Saved {len(labels_dict)} labeled glyphs:')
        for idx, piece in sorted(labels_dict.items()):
            result = all_moves[idx]
            print(f'   {piece} (P{result["page"]}: {result["move_text"]})')
    
    save_button.on_click(on_save_clicked)
    
    # Display in grid
    print('👁️  VISUALLY INSPECT each move image and label the piece (K, Q, R, B, N, P or [None]):\n')
    
    for row in range(n_rows):
        for col in range(n_cols):
            idx = row * n_cols + col
            if idx < len(glyph_items):
                item = glyph_items[idx]
                crop = item['result']['crop_image']
                
                # Show image and dropdown
                fig, ax = plt.subplots(figsize=(1.8, 2))
                if crop:
                    arr = np.array(crop, dtype=np.float32) / 255.0
                    ax.imshow(arr, cmap='gray')
                ax.set_title(f"P{item['page']}: {item['move']}\n({item['piece_from_text']})", fontsize=8, pad=5)
                ax.axis('off')
                plt.tight_layout()
                plt.show()
                
                display(item['dropdown'])
                print()
    
    print('\n' + '='*60)
    display(save_button)
    print('='*60)

In [ ]:
# ── Build training dataset from YOUR manually-labeled moves ────────────────
print('Building training dataset from your labeled moves...\n')

if not labels_dict:
    print('❌ No labels found. Click "Save Labels" in labeling cell first.')
else:
    labeled_results = []
    for idx, piece_label in labels_dict.items():
        if idx < len(all_moves):
            result = all_moves[idx].copy()
            result['inferred_piece'] = piece_label  # Use YOUR label (K, Q, R, B, N, P)
            labeled_results.append(result)
    
    print(f'✅ Using {len(labeled_results)} manually-labeled moves:\n')
    
    piece_counts = defaultdict(int)
    for result in labeled_results:
        piece_counts[result['inferred_piece']] += 1
    
    for piece in ['K', 'Q', 'R', 'B', 'N', 'P']:
        count = piece_counts[piece]
        status = '✅' if count > 0 else '  '
        if count > 0:
            print(f'  {status} {piece}: {count}')
    
    print()
    
    # Check if we have enough samples per piece
    min_samples = 3
    low_samples = [p for p in ['K', 'Q', 'R', 'B', 'N'] if piece_counts[p] < min_samples]
    if low_samples:
        print(f'⚠️  Low samples for: {", ".join(low_samples)}. Consider labeling more moves.')
    
    # Store for training step
    extraction_results_labeled = labeled_results

In [ ]:
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import random

def augment_glyph(pil_img, n, target_size=32):
    """Generate n augmented versions of a glyph image for training."""
    results = []
    
    # Resize to target size
    if pil_img.size[0] == 0 or pil_img.size[1] == 0:
        return []
    
    base = pil_img.resize((target_size, target_size), Image.LANCZOS)
    
    for _ in range(n):
        img = base.copy()
        
        # Random rotation ±12°
        angle = random.uniform(-12, 12)
        img = img.rotate(angle, resample=Image.BICUBIC, fillcolor=255)
        
        # Random scale 0.80 – 1.20
        scale  = random.uniform(0.80, 1.20)
        new_sz = max(4, int(target_size * scale))
        img    = img.resize((new_sz, new_sz), Image.LANCZOS)
        canvas = Image.new('L', (target_size, target_size), 255)
        off    = (target_size - new_sz) // 2
        paste_box = (max(0, off), max(0, off),
                     min(target_size, new_sz + off), min(target_size, new_sz + off))
        crop_box = (max(0, -off), max(0, -off),
                    min(new_sz, target_size - off), min(new_sz, target_size - off))
        img_crop = img.crop(crop_box)
        canvas.paste(img_crop, paste_box)
        img = canvas
        
        # Random horizontal flip
        if random.random() > 0.5:
            img = ImageOps.mirror(img)
        
        # Occasional subtle blur
        if random.random() > 0.8:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 0.6)))
        
        # Convert to float32 in [0, 1]
        arr = np.array(img, dtype=np.float32) / 255.0
        
        # Random noise
        arr += np.random.normal(0, 0.03, arr.shape)
        
        # Random brightness
        brightness = random.uniform(0.85, 1.15)
        arr = arr * brightness + random.uniform(-0.04, 0.04)
        arr = np.clip(arr, 0.0, 1.0)
        
        results.append(arr)
    
    return results

# ── Build training dataset from YOUR manually-labeled glyphs ────────────────
# Use extraction_results_labeled if available (from manual labeling), otherwise fall back
data_to_use = extraction_results_labeled if 'extraction_results_labeled' in locals() else extraction_results

print(f'Building training dataset from {len(data_to_use)} glyphs...\n')

if not data_to_use:
    print('❌ No glyphs to process. Run extraction and labeling steps first.')
else:
    X, y = [], []
    samples_per_piece = defaultdict(int)
    
    for result in data_to_use:
        crop_image = result['crop_image']
        piece = result['inferred_piece']
        
        # Validate piece is in CLASS_NAMES
        if piece not in CLASS_NAMES:
            print(f'⚠️  Skipping invalid piece: {piece}')
            continue
        
        piece_idx = CLASS_NAMES.index(piece)
        
        # Augment each glyph
        augmented = augment_glyph(crop_image, AUGMENT_PER_GLYPH)
        for aug_array in augmented:
            X.append(aug_array)
            y.append(piece_idx)
            samples_per_piece[piece] += 1
    
    if len(X) > 0:
        X = np.array(X)[..., np.newaxis]  # (N, 32, 32, 1)
        y = np.array(y)
        
        print(f'✅ Generated {len(X)} training samples from {len(data_to_use)} glyphs:')
        for piece in CLASS_NAMES:
            count = samples_per_piece.get(piece, 0)
            status = '✅' if count >= MIN_GLYPH_SAMPLES else '⚠'
            if count > 0:
                print(f'   {status} {piece}: {count}')
        
        if len(X) < 100:
            print(f'\n⚠ Warning: Only {len(X)} samples. Consider extracting more glyphs.')
    else:
        print('❌ No augmented samples generated.')

In [ ]:
import matplotlib.pyplot as plt

if 'X' not in locals() or len(X) == 0:
    print('❌ No training data yet.')
    print('   Run these steps in order:')
    print('   1. Step 3 — Extract glyphs')
    print('   2. Labeling UI — Label each glyph')
    print('   3. Build Dataset — Process labels')
    print('   4. Augmentation cell — Create training data (X, y)')
    print('   5. Then come back here to see samples')
else:
    fig, axes = plt.subplots(5, 10, figsize=(14, 9))
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        if cls_idx >= 5:  # Only show K, Q, R, B, N (5 classes, no P)
            break
        samples = X[y == cls_idx][:10]
        for col, img in enumerate(samples):
            ax = axes[cls_idx][col]
            ax.imshow(img.squeeze(), cmap='gray', vmin=0, vmax=1)
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(cls_name, fontsize=11, rotation=0, labelpad=20, va='center')
    plt.suptitle('Sample training images (10 per class)', fontsize=12)
    plt.tight_layout()
    plt.show()

## Step 4 — Train CNN on extracted glyphs

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Check if we have training data
if 'X' not in locals() or len(X) == 0:
    print('❌ No training data. Run the glyph extraction step above first.')
else:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
    )
    print(f'Train: {len(X_train)}  Val: {len(X_val)}')
    
    # ── Model ──────────────────────────────────────────────────────────────────
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D(2),
    
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D(2),
    
        tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
    
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
    ], name='figurine_classifier')
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.summary()
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True,
                                         monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5,
                                             monitor='val_accuracy'),
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    
    val_acc = max(history.history['val_accuracy'])
    print(f'\n✅ Best val accuracy: {val_acc:.1%}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('epoch')
ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('epoch')
plt.tight_layout(); plt.show()

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)
plt.title('Validation confusion matrix')
plt.tight_layout(); plt.show()

## Step 5 — Export TFLite model

In [ ]:
if 'model' not in locals():
    print('❌ Model not trained. Run the training step first.')
else:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # float32 — compatible with all TFLite runtimes, no calibration needed.
    tflite_model = converter.convert()
    
    with open(TFLITE_PATH, 'wb') as f:
        f.write(tflite_model)
    
    size_kb = os.path.getsize(TFLITE_PATH) / 1024
    print(f'✅ Saved: {TFLITE_PATH}  ({size_kb:.0f} KB)')
    
    # ── Sanity-check on the TFLite model ──────────────────────────────────
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    print(f'   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
    print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')
    
    # Run one test image per class (if available).
    print('\nClass-level spot check:')
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        samples = X_val[y_val == cls_idx]
        if len(samples) > 0:
            sample = samples[0:1].astype(np.float32)
            interp.set_tensor(inp['index'], sample)
            interp.invoke()
            probs = interp.get_tensor(out['index'])[0]
            pred  = CLASS_NAMES[np.argmax(probs)]
            conf  = probs.max()
            ok    = '✅' if pred == cls_name else '⚠'
            print(f'  {ok} true={cls_name}  pred={pred}  conf={conf:.1%}')
        else:
            print(f'  ⚠ {cls_name}: no validation samples')

In [ ]:
from google.colab import files

if os.path.exists(TFLITE_PATH):
    files.download(TFLITE_PATH)
    print(f'✅ Downloaded {TFLITE_PATH}')
else:
    print(f'❌ Model file not found: {TFLITE_PATH}. Run export step first.')

## Flutter integration

1. **Before running this notebook:**
   - Upload your chess PDF to Google Drive (the one you want to read with the app)
   - Edit `PDF_PATH` in Step 1 to point to your PDF
   - Run Step 2 to extract glyphs and see which characters are piece symbols
   - Create a `GLYPH_MAPPING` in Step 3a mapping each character to its piece letter

2. **After training:**
   - Download `figurine_classifier.tflite` from the notebook
   - Copy it to `assets/models/figurine_classifier.tflite` in your Flutter project
   - Declare the asset in `pubspec.yaml` (already done)
   - Rebuild and run the app

3. **In the app:**
   - The `FigurineClassifier` loads the model at startup
   - For each PDF page: renders it, extracts text with bboxes
   - For non-ASCII characters (piece glyphs), crops from the rendered image
   - Runs this model to get confidence scores
   - Builds `fontMap` from char → piece prediction
   - Passes `fontMap` to `MoveParser` for accurate move parsing

4. **Model expectations:**
   - Input: Float32List of length 1024 (32×32 pixels, values in \[0,1\])
   - Output: Float32List of length 6 (probabilities for K, Q, R, B, N, P)
   - Use `argmax(output)` to get predicted piece class